In [1]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0.6
)

## **Output produce in Stream manner**

In [ ]:
for chunk in llm.stream("write 1000 word paragraph about Agentic Ai"):  
    print(chunk.text(), end="")

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
embedding_len = len(embeddings.embed_query("hello"))
embedding_len

768

## **Data ingestion**
[LLMs Powered autonomous agents](https://lilianweng.github.io/posts/2023-06-23-agent/)

In [9]:
url = "https://lilianweng.github.io/posts/2023-06-23-agent/"

from langchain_community.document_loaders import WebBaseLoader

web_load = WebBaseLoader(url)
data = web_load.load()

In [ ]:
## check the actual data

data[0].page_content

In [13]:
data[0].metadata

{'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/',
 'title': "LLM Powered Autonomous Agents | Lil'Log",
 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final results.\n\n\nMemory\

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

all_chunks = splitter.split_documents(documents=data)
len(all_chunks)

63

## **Extract data form the multiple URL or LIST of url**

In [18]:
url_list = [
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2020-04-07-the-transformer-family/"
]

In [20]:
docs = [WebBaseLoader(url).load() for url in url_list]
docs

[[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/', 'title': "Prompt Engineering | Lil'Log", 'description': 'Prompt Engineering, also known as In-Context Prompting, refers to methods for how to communicate with LLM to steer its behavior for desired outcomes without updating the model weights. It is an empirical science and the effect of prompt engineering methods can vary a lot among models, thus requiring heavy experimentation and heuristics.\nThis post only focuses on prompt engineering for autoregressive language models, so nothing with Cloze tests, image generation or multimodality models. At its core, the goal of prompt engineering is about alignment and model steerability. Check my previous post on controllable text generation.', 'language': 'en'}, page_content='\n\n\n\n\n\nPrompt Engineering | Lil\'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLil\'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n|\n\n\n\

In [21]:
bpe_token_text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 100,
    chunk_overlap = 25
)

## List compression

In [28]:
docs = [item for sublist in docs for item in sublist]
docs

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/', 'title': "Prompt Engineering | Lil'Log", 'description': 'Prompt Engineering, also known as In-Context Prompting, refers to methods for how to communicate with LLM to steer its behavior for desired outcomes without updating the model weights. It is an empirical science and the effect of prompt engineering methods can vary a lot among models, thus requiring heavy experimentation and heuristics.\nThis post only focuses on prompt engineering for autoregressive language models, so nothing with Cloze tests, image generation or multimodality models. At its core, the goal of prompt engineering is about alignment and model steerability. Check my previous post on controllable text generation.', 'language': 'en'}, page_content='\n\n\n\n\n\nPrompt Engineering | Lil\'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLil\'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n|\n\n\n\n

### ***BPE Embedding its Form OpenAI***
- since i've openai api key so i use `HF` embeddings so i use only the RecursiveCharTextSplitter only

In [29]:
prompt_n_transformer_docs = bpe_token_text_splitter.split_documents(documents=docs)

In [31]:
prompt_transformer_docs = splitter.split_documents(documents=docs)
len(prompt_transformer_docs)

99

## **Create Vector DB**

In [32]:
from langchain_community.vectorstores import Chroma

In [40]:
vectorstore_1 = Chroma.from_documents(
    documents= all_chunks,
    collection_name="autonomus-agent",
    embedding=embeddings


)

retriever_1 = vectorstore_1.as_retriever()

In [41]:
vectorstore_2 = Chroma.from_documents(
    documents=prompt_transformer_docs,
    embedding=embeddings,
    collection_name="prompt-transformer"
)

retriever_2 = vectorstore_2.as_retriever()

In [42]:

retriever_1.invoke("what do you mean be autonomus agent?")

[Document(metadata={'language': 'en', 'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the qua

In [37]:
from langchain.tools.retriever import create_retriever_tool

In [43]:
retriever_tool_1 = create_retriever_tool(
    retriever=retriever_1,
    name="all about autonomous agent",
    description="LLM Powered Autonomous Agents related blog. and there have info about related to agent."
)

retriever_tool_2 = create_retriever_tool(
    retriever=retriever_2,
    name="prompt and transformer-tools",
    description="this tools have prompt and transformer related info."
)

In [47]:
tools = [retriever_tool_1, retriever_tool_2]

In [48]:
from langgraph.prebuilt import ToolNode
tools = ToolNode(tools)
tools

tools(tags=None, recurse=True, explode_args=False, func_accepts_config=True, func_accepts={'store': ('__pregel_store', None)}, tools_by_name={'all about autonomous agent': Tool(name='all about autonomous agent', description='LLM Powered Autonomous Agents related blog. and there have info about related to agent.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=functools.partial(<function _get_relevant_documents at 0x000002B62E44ACA0>, retriever=VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000002B63329AED0>, search_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template='{page_content}'), document_separator='\n\n', response_format='content'), coroutine=functools.partial(<function _aget_relevant_documents at 0x000002B62E44BCE0>, retriever=VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vector